# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:

import pandas as pd
from pathlib import Path
import subprocess

# GitHub repository path
repo_path = Path("/content/flyrank-ml-workspace2")

# Clone repository if not already present
if not repo_path.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/suyashverma0/flyrank-ml-workspace2.git",
            str(repo_path)
        ],
        check=True
    )

# Find dataset
matches = list(
    repo_path.rglob("content_refresh_anonymized.csv")
)

if not matches:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv not found!"
    )

# Load dataset
data_path = matches[0]
df = pd.read_csv(data_path)

# Verify dataset
print("Dataset loaded successfully!")
print("Dataset location:", data_path)
print("Dataset shape:", df.shape)

display(df.head())


In [ ]:
research_question = (
    "Can content metadata help prioritize content items for human review "
    "using a historical declining-content proxy?"
)

decision_supported = (
    "Prioritize which content items an analyst should inspect first."
)

print("Research question:", research_question)
print("Decision supported:", decision_supported)

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import subprocess

repo_path = Path("/content/flyrank-ml-workspace2")

if not repo_path.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/suyashverma0/flyrank-ml-workspace2.git",
        str(repo_path)
    ], check=True)

matches = list(repo_path.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv not found in the repository."
    )

data_path = matches[0]
df = pd.read_csv(data_path)

print("Dataset file:", data_path.name)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn list:")
print(df.columns.tolist())

print("\nMissing values:")
display(
    df.isna().sum()
    .sort_values(ascending=False)
    .head(15)
    .rename("missing_count")
    .to_frame()
)

assert "trend_direction" in df.columns
assert "client_id" in df.columns
assert "content_id" in df.columns

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Create historical proxy label
df = df.copy()

df["is_declining_proxy"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Model features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent"
]

feature_columns = numeric_features + categorical_features

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(f"Missing model features: {missing_features}")

X = df[feature_columns].copy()
y = df["is_declining_proxy"].copy()
groups = df["client_id"].copy()

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients)

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print("Client overlap:", len(train_clients.intersection(test_clients)))

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Model training completed.")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# Baseline thresholds are calculated using training records only
train_rows = df.iloc[train_idx]
test_rows = df.iloc[test_idx].copy()

ctr_train = pd.to_numeric(
    train_rows["ctr"], errors="coerce"
)

engagement_train = pd.to_numeric(
    train_rows["engagement_rate"], errors="coerce"
)

ctr_threshold = ctr_train.quantile(0.25)
engagement_threshold = engagement_train.quantile(0.25)

# Calculate baseline score on the same test rows
baseline_score = pd.Series(0, index=test_rows.index, dtype=int)

test_ctr = pd.to_numeric(
    test_rows["ctr"], errors="coerce"
)

test_engagement = pd.to_numeric(
    test_rows["engagement_rate"], errors="coerce"
)

test_age = pd.to_numeric(
    test_rows["days_since_last_update"], errors="coerce"
)

test_position = pd.to_numeric(
    test_rows["avg_position"], errors="coerce"
)

baseline_score += (
    test_ctr.notna() & (test_ctr <= ctr_threshold)
).astype(int)

baseline_score += (
    test_engagement.notna()
    & (test_engagement <= engagement_threshold)
).astype(int)

baseline_score += (
    test_age.notna() & (test_age >= 180)
).astype(int)

baseline_score += (
    test_position.notna() & (test_position > 20)
).astype(int)

# Ranking scores
results = pd.DataFrame({
    "actual_proxy": y_test.values,
    "model_score": model_scores,
    "baseline_score": baseline_score.values
}, index=test_rows.index)

def precision_at_k(y_true, scores, k):
    ranked_indices = np.argsort(-np.asarray(scores))[:k]
    selected = np.asarray(y_true)[ranked_indices]
    return selected.mean() if len(selected) else np.nan

evaluation_rows = []

for method, score_col in [
    ("Rule-based baseline", "baseline_score"),
    ("Logistic Regression", "model_score")
]:
    row = {"Method": method}

    for k in [10, 50, 100]:
        row[f"Precision@{k}"] = precision_at_k(
            results["actual_proxy"],
            results[score_col],
            k
        )

    evaluation_rows.append(row)

evaluation_table = pd.DataFrame(evaluation_rows)

print("Positive proxy rate in test set:",
      round(results["actual_proxy"].mean(), 4))

display(evaluation_table)

# Save exact comparison table
output_dir = repo_path / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

evaluation_table.to_csv(
    output_dir / "capstone_model_vs_baseline.csv",
    index=False
)

## 5. Limitations

*What this work cannot claim.*

In [ ]:
limitations = [
    "Target is a historical trend proxy, not a verified future outcome.",
    "Anonymized dataset may not represent all clients or content types.",
    "Client-grouped validation does not replace time-based validation.",
    "Missing values and data availability may affect model results.",
    "Baseline uses a small set of manually selected rules.",
    "No causal effect of content refresh is measured."
]

for i, limitation in enumerate(limitations, start=1):
    print(f"{i}. {limitation}")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# Build recommendation queue from the same held-out test records
recommendations = test_rows.copy()

recommendations["model_score"] = model_scores
recommendations["baseline_score"] = baseline_score.values

def make_reason_codes(row):
    reasons = []

    ctr = pd.to_numeric(
        pd.Series([row.get("ctr")]), errors="coerce"
    ).iloc[0]

    engagement = pd.to_numeric(
        pd.Series([row.get("engagement_rate")]), errors="coerce"
    ).iloc[0]

    age = pd.to_numeric(
        pd.Series([row.get("days_since_last_update")]), errors="coerce"
    ).iloc[0]

    position = pd.to_numeric(
        pd.Series([row.get("avg_position")]), errors="coerce"
    ).iloc[0]

    if pd.notna(ctr) and ctr <= ctr_threshold:
        reasons.append("Low CTR")

    if pd.notna(engagement) and engagement <= eng_threshold:
        reasons.append("Low engagement")

    if pd.notna(age) and age >= 180:
        reasons.append("Not updated for 180+ days")

    if pd.notna(position) and position > 20:
        reasons.append("Weak average position")

    return "; ".join(reasons) if reasons else "No listed warning signal"

recommendations["reason_codes"] = recommendations.apply(
    make_reason_codes,
    axis=1
)

recommendations = recommendations.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

recommendations.insert(
    0, "priority_rank", range(1, len(recommendations) + 1)
)

# Public-safe output: remove client identifiers and private fields
public_queue_columns = [
    "priority_rank",
    "content_id",
    "model_score",
    "reason_codes",
    "ctr",
    "engagement_rate",
    "days_since_last_update",
    "avg_position"
]

recommendation_export = recommendations[
    [col for col in public_queue_columns if col in recommendations.columns]
].copy()

display(recommendation_export.head(20))

recommendation_export.to_csv(
    output_dir / "capstone_ranked_recommendations.csv",
    index=False
)

print("Ranked recommendations exported.")

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import matplotlib.pyplot as plt

# Chart 1: Model vs baseline comparison
plot_table = evaluation_table.set_index("Method")

plot_table.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.ylabel("Precision")
plt.title("Model vs Rule-Based Baseline")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    output_dir / "capstone_model_vs_baseline.png",
    dpi=150
)
plt.show()

# Chart 2: Model score distribution
plt.figure(figsize=(8, 5))
plt.hist(recommendations["model_score"], bins=20)
plt.xlabel("Model score")
plt.ylabel("Number of content items")
plt.title("Distribution of Model Scores")
plt.tight_layout()

plt.savefig(
    output_dir / "capstone_model_score_distribution.png",
    dpi=150
)
plt.show()

# Chart 3: Most frequent reason codes
reason_counts = (
    recommendations["reason_codes"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(9, 5))
reason_counts.plot(kind="barh")
plt.xlabel("Number of content items")
plt.ylabel("Reason code")
plt.title("Most Common Recommendation Reason Codes")
plt.tight_layout()

plt.savefig(
    output_dir / "capstone_reason_codes.png",
    dpi=150
)
plt.show()

# Export a reproducibility summary
run_summary = pd.DataFrame({
    "item": [
        "Dataset rows",
        "Dataset columns",
        "Training rows",
        "Test rows",
        "Training clients",
        "Test clients",
        "Client overlap",
        "Positive proxy rate in test set"
    ],
    "value": [
        len(df),
        df.shape[1],
        len(train_idx),
        len(test_idx),
        len(train_clients),
        len(test_clients),
        len(train_clients.intersection(test_clients)),
        round(results["actual_proxy"].mean(), 4)
    ]
})

run_summary.to_csv(
    output_dir / "capstone_run_summary.csv",
    index=False
)

print("Capstone artifacts:")
for file in sorted(output_dir.glob("capstone_*")):
    print(file.name)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
